## 02 — Préparation des données

Cellule 1 — Chargement :

In [1]:
import pandas as pd

df_petit = pd.read_csv(
    "/content/drive/MyDrive/PFE_Trafic/data/df_petit.csv"
)
print("Données chargées :", df_petit.shape)
df_petit.head()

Données chargées : (1000, 551)


,Page,2015-07-01,2015-07-02,2015-07-03,2015-07-04,2015-07-05,2015-07-06,2015-07-07,2015-07-08,2015-07-09,...,2016-12-22,2016-12-23,2016-12-24,2016-12-25,2016-12-26,2016-12-27,2016-12-28,2016-12-29,2016-12-30,2016-12-31
0,Phabricator/Project_management_www.mediawiki.o...,6.0,6.0,4.0,6.0,8.0,6.0,4.0,0.0,2.0,...,6.0,6.0,11.0,4.0,6.0,5.0,7.0,6.0,6.0,9.0
1,Now_You_See_Me_es.wikipedia.org_desktop_all-ag...,242.0,271.0,309.0,227.0,321.0,311.0,242.0,236.0,243.0,...,231.0,222.0,193.0,229.0,334.0,316.0,324.0,268.0,201.0,190.0
2,Zürich_Hackathon_2014_www.mediawiki.org_all-ac...,3.0,19.0,19.0,30.0,21.0,24.0,17.0,178.0,40.0,...,6.0,7.0,4.0,8.0,2.0,4.0,9.0,4.0,11.0,12.0
3,Érythrée_fr.wikipedia.org_desktop_all-agents,672.0,513.0,774.0,1164.0,546.0,755.0,555.0,494.0,4801.0,...,308.0,294.0,358.0,204.0,323.0,438.0,345.0,299.0,306.0,211.0
4,Metallica_es.wikipedia.org_all-access_all-agents,1534.0,1644.0,1704.0,1569.0,1534.0,1577.0,1608.0,1731.0,1919.0,...,2367.0,2259.0,2229.0,2070.0,2774.0,2552.0,2524.0,2358.0,2291.0,2153.0


Cellule 2 — Conversion format long :

In [2]:
df_long = df_petit.melt(id_vars="Page", var_name="date", value_name="views")
df_long["date"] = pd.to_datetime(df_long["date"])
df_long = df_long.sort_values(["Page", "date"]).reset_index(drop=True)

print("Format long :", df_long.shape)

Format long : (550000, 3)


Cellule 3 — Création des features :

In [3]:
df_long["lag_1"]  = df_long.groupby("Page")["views"].shift(1)
df_long["lag_7"]  = df_long.groupby("Page")["views"].shift(7)
df_long["lag_14"] = df_long.groupby("Page")["views"].shift(14)

df_long["mean_7"] = df_long.groupby("Page")["views"].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

df_long["day_of_week"] = df_long["date"].dt.dayofweek
df_long["month"]       = df_long["date"].dt.month

df_features = df_long.dropna().reset_index(drop=True)

print("Lignes après nettoyage :", len(df_features))
df_features.head()

Lignes après nettoyage : 536000


,Page,date,views,lag_1,lag_7,lag_14,mean_7,day_of_week,month
0,15._November_de.wikipedia.org_desktop_all-agents,2015-07-15,27.0,21.0,27.0,32.0,30.142857,2,7
1,15._November_de.wikipedia.org_desktop_all-agents,2015-07-16,20.0,27.0,19.0,26.0,30.142857,3,7
2,15._November_de.wikipedia.org_desktop_all-agents,2015-07-17,23.0,20.0,21.0,22.0,30.285714,4,7
3,15._November_de.wikipedia.org_desktop_all-agents,2015-07-18,23.0,23.0,21.0,22.0,30.571429,5,7
4,15._November_de.wikipedia.org_desktop_all-agents,2015-07-19,27.0,23.0,64.0,29.0,30.857143,6,7


Cellule 4 — Sauvegarde :

In [4]:
df_features.to_csv(
    "/content/drive/MyDrive/PFE_Trafic/data/df_features.csv",
    index=False
)
print("df_features sauvegardé :", len(df_features), "lignes")

df_features sauvegardé : 536000 lignes
